# POUNDERS with a length ladder (no adaptive shots)

Plain **POUNDERS + FPR**, run **iteratively over increasing max-length `L`** exactly like pyGSTi's
`StandardGST`/LM does — instead of fitting all circuits at once. At each stage we fit the **cumulative**
circuit set (all circuits up to `L`), **warm-started from the previous stage's estimate**, so the short
circuits condition the problem before the long ones sharpen it.

**No adaptive shot allocation** here — this is the fixed-shots `fixed_fpr` optimizer, just *staged*.
Data is sampled **once** for all circuits (measure-once, fit-iteratively), so total shots == `fixed_fpr`
at the same `shots/circuit`; only the *fit procedure* differs.

> Needs PyROL (POUNDERS) — run in Docker. The staging/plumbing is standard pyGSTi; only `pounders.pouders`
> needs PyROL.

In [ ]:
import sys, json, importlib
from pathlib import Path
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidates = [
    Path.cwd(), Path.cwd() / 'seed_sweep_experiments',
    Path.cwd() / 'GST_POUNDERS' / 'seed_sweep_experiments',
    Path('/workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments'),
]
EXPERIMENT_DIR = next((p.resolve() for p in candidates if (p / 'gst_seed_experiment.py').exists()), None)
if EXPERIMENT_DIR is None:
    raise FileNotFoundError('Could not locate gst_seed_experiment.py')
if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))
import gst_seed_experiment as gse
from gst_seed_experiment import ExperimentConfig, GSTProblem

config = ExperimentConfig.from_json(EXPERIMENT_DIR / 'experiment_config.json')
INF  = 'mean_gate_entanglement_infidelity_to_truth'
SPAM = 'mean_spam_vector_l2_error_to_truth'
print('Experiment dir:', EXPERIMENT_DIR)
print('noise_model:', config.noise_model, '| max_lengths:', config.max_lengths,
      '| fixed_fpr_shots:', config.fixed_fpr_shots)

In [ ]:
# ------------------------- knobs -------------------------
SEEDS        = [1010]     # data seeds
SHOTS        = int(config.fixed_fpr_shots)   # shots/circuit (measured once for ALL circuits)
MAX_LENGTHS  = list(config.max_lengths)      # the ladder: [1, 2, 4, 8, 16, 32, 64]
STAGE_NFMAX  = 200                           # POUNDERS evals PER stage (total ~= 7 * this)
RESULTS_DIR  = EXPERIMENT_DIR / 'pounders_ladder'
FORCE        = True
print('ladder stages:', MAX_LENGTHS, '| shots/circuit:', SHOTS, '| nfmax/stage:', STAGE_NFMAX)

In [ ]:
# ------------------------- ladder machinery -------------------------
import pygsti

def restrict_dataset(master_ds, circuits):
    """A DataSet holding only `circuits`, counts copied from the master (shared data)."""
    ds = pygsti.data.DataSet(outcome_labels=list(master_ds.outcome_labels))
    for c in circuits:
        ds.add_count_dict(c, dict(master_ds[c].counts))
    ds.done_adding_data()
    return ds

def run_pounders_stage(problem, cfg, shots, x0, nfmax):
    """One plain POUNDERS+FPR solve on `problem` (dataset already set), warm-started at x0."""
    import gradient_pounders, general_h_funs
    pounders = importlib.reload(gradient_pounders)
    ghf = importlib.reload(general_h_funs)
    pounders.PYROL_INNER_ITERATION_LIMIT = int(cfg.pyrol_max_iters)
    if cfg.require_pyrol and (importlib.util.find_spec('pyrol') is None
                              and importlib.util.find_spec('ROL') is None):
        raise ModuleNotFoundError('PyROL/ROL not importable - run in the Docker image.')
    problem.x0 = np.asarray(x0, dtype=float).reshape(-1)
    problem.shots_per_circuit = problem.normalize_shots(shots)
    fpr_reduction = gse._build_fpr(problem, cfg)
    lower = np.full(problem.n, float(cfg.lower_bound))
    upper = np.full(problem.n, float(cfg.upper_bound))
    X, F, J, flag, xkin = pounders.pouders(
        problem.oracle, problem.x0.reshape(1, -1), problem.n, int(nfmax),
        float(cfg.gtol), float(cfg.initial_delta), problem.m, lower, upper,
        gse.PoundersLogger(enabled=False), spsolver=3,
        hfun=ghf.h_leastsquares, combinemodels=ghf.combine_leastsquares,
        fpr_reduction=fpr_reduction,
        residuals_per_circuit=problem.outcomes_per_circuit,
        shots_per_circuit=int(shots),
        fpr_use_union_mask=bool(cfg.use_fpr_union_mask),
        rho_uses_full_objective=bool(cfg.rho_uses_full_objective),
        iter_callback=lambda state: None,
    )
    selected = set()                     # FPR-selected circuits this stage (for accounted shots)
    for h in (getattr(fpr_reduction, 'history', None) or []):
        idxs = h.get('selected_circuit_indices') if isinstance(h, dict) else None
        if isinstance(idxs, str):
            import ast
            try: idxs = ast.literal_eval(idxs)
            except Exception: idxs = None
        for i in (idxs or []):
            if 0 <= int(i) < len(problem.circuits):
                selected.add(problem.circuits[int(i)])
    x_best = np.asarray(X[int(xkin)], dtype=float).reshape(-1)
    return x_best, problem.copy_model_at_x(x_best), flag, selected

def run_ladder(cfg, seed, shots, max_lengths, stage_nfmax):
    """Stage POUNDERS over increasing L, warm-started, on one shared measure-once dataset."""
    max_lengths = list(max_lengths)
    # 1. sample the full dataset ONCE (all circuits) -> the master
    full = GSTProblem(replace(cfg, max_lengths=tuple(max_lengths)), seed)
    full.set_uniform_dataset(int(shots))
    master_ds = full.dataset
    total_shots = int(len(full.circuits) * int(shots))
    # 2. climb the ladder, warm-starting each stage from the last estimate
    prev_x, traj, revealed = None, [], set()
    for k, L in enumerate(max_lengths):
        stage_cfg = replace(cfg, max_lengths=tuple(max_lengths[:k + 1]))
        prob = GSTProblem(stage_cfg, seed)
        prob.dataset = restrict_dataset(master_ds, prob.circuits)   # shared data, this stage's circuits
        x0 = prev_x if prev_x is not None else prob.x0
        x_best, fit, flag, sel = run_pounders_stage(prob, stage_cfg, shots, x0, stage_nfmax)
        revealed |= sel
        st, _, _ = prob.aligned_error_metrics(fit, prob.truth_model, 'truth')
        traj.append({'stage': k, 'max_length': int(L), 'n_circuits': len(prob.circuits),
                     INF: float(st[INF]), SPAM: float(st.get(SPAM, float('nan'))), 'flag': str(flag)})
        print(f'  seed {seed} stage {k} (L={L}, {len(prob.circuits)} circ): infid={st[INF]:.3e}')
        prev_x = x_best
    accounted = int(len(revealed) * int(shots)) if revealed else int(total_shots)
    return {'trajectory': traj, 'final_x': prev_x, 'total_shots': total_shots,
            'accounted_revealed_shots': accounted, 'revealed_circuits': len(revealed),
            INF: traj[-1][INF], SPAM: traj[-1][SPAM]}

In [ ]:
# ------------------------- run over seeds -------------------------
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
rows = []
for seed in SEEDS:
    seed_dir = RESULTS_DIR / f'seed_{seed:06d}'
    if (seed_dir / 'ladder_summary.json').exists() and not FORCE:
        print('SKIP seed', seed)
        rows.append(json.loads((seed_dir / 'ladder_summary.json').read_text()))
        continue
    print('RUN ladder seed', seed)
    res = run_ladder(config, seed, SHOTS, MAX_LENGTHS, STAGE_NFMAX)
    seed_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(res['trajectory']).to_csv(seed_dir / 'ladder_trajectory.csv', index=False)
    np.save(seed_dir / 'ladder_final_x.npy', res['final_x'])
    summ = {'seed': seed, 'shots_per_circuit': SHOTS, 'total_shots': res['total_shots'],
            'accounted_revealed_shots': res.get('accounted_revealed_shots', res['total_shots']),
            'revealed_circuits': res.get('revealed_circuits', 0),
            INF: res[INF], SPAM: res[SPAM]}
    (seed_dir / 'ladder_summary.json').write_text(json.dumps(summ, indent=2))
    rows.append(summ)
    pd.DataFrame(rows).to_csv(RESULTS_DIR / 'ladder_summary.csv', index=False)

summary = pd.DataFrame(rows)
summary.to_csv(RESULTS_DIR / 'ladder_summary.csv', index=False)
print()
print(summary.to_string(index=False))

## Per-stage convergence (infidelity vs max-length L)

In [ ]:
frames = []
for p in sorted(RESULTS_DIR.glob('seed_*/ladder_trajectory.csv')):
    fr = pd.read_csv(p); fr['seed'] = int(p.parent.name.split('_')[-1]); frames.append(fr)
traj = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if traj.empty:
    print('No ladder_trajectory.csv yet - run the ladder first.')
else:
    fig, ax = plt.subplots(figsize=(9, 6))
    for seed, g in traj.groupby('seed'):
        g = g.sort_values('max_length')
        ax.plot(g['max_length'], g[INF], color='0.75', lw=1.0, alpha=0.7, zorder=1)
    grp = traj.groupby('max_length')[INF]
    x = np.array(sorted(traj['max_length'].unique()), dtype=float)
    med = grp.median().reindex(x).to_numpy(float)
    q25 = grp.quantile(0.25).reindex(x).to_numpy(float)
    q75 = grp.quantile(0.75).reindex(x).to_numpy(float)
    ax.plot(x, med, 'o-', color='#0072B2', lw=2.4, ms=8, label='median', zorder=3)
    ax.fill_between(x, q25, q75, color='#0072B2', alpha=0.2, zorder=2)
    ax.axhline(1e-4, ls='--', color='black', lw=1.1, label='target 1e-4')
    ax.set_xscale('log', base=2); ax.set_yscale('log')
    ax.set_xticks(x); ax.set_xticklabels([int(v) for v in x])
    ax.set_xlabel('max circuit length L (ladder stage)')
    ax.set_ylabel('mean gate infidelity to truth (log)')
    ax.set_title('POUNDERS length-ladder: infidelity vs stage (median +/- IQR)')
    ax.grid(alpha=0.25, which='both'); ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'ladder_convergence.png', dpi=200, bbox_inches='tight')
    plt.show()

## Final result table

In [ ]:
if not summary.empty:
    print('final ladder infidelity (to truth) per seed:')
    print(summary[['seed', 'total_shots', INF, SPAM]].to_string(index=False))
    print()
    print(f'median final infidelity: {summary[INF].median():.3e}   '
          f'(target 1e-4; # seeds below: {(summary[INF] < 1e-4).sum()}/{len(summary)})')

## Compare: ladder vs `fixed_fpr` (all-at-once) vs LM

All three fit the **same shot-matched dataset** (all circuits x `SHOTS`, same seed -> same noise realization),
so only the **fit procedure** differs: staged POUNDERS (this ladder), one-shot POUNDERS+FPR (`fixed_fpr`),
and pyGSTi's LM (`StandardGST`, which ladders internally). Run in Docker (fixed_fpr needs PyROL; LM is pure pyGSTi).

In [ ]:
from gst_seed_experiment import run_one_experiment

LM_MODES, LM_MAXITER = 'CPTPLND', 300      # pyGSTi LM baseline

def run_or_load_fixed_fpr(seed, shots, out_dir, force):
    if (out_dir / 'summary.json').exists() and not force:
        return json.loads((out_dir / 'summary.json').read_text())
    return run_one_experiment(config=replace(config, fixed_fpr_shots=int(shots)),
                              data_seed=seed, method='fixed_fpr', output_dir=out_dir)

def fit_or_load_lm(seed, shots, lm_dir, force):
    if (lm_dir / 'summary.json').exists() and (lm_dir / 'lm_trajectory.csv').exists() and not force:
        return json.loads((lm_dir / 'summary.json').read_text())
    import re, pygsti
    from pygsti.optimize import SimplerLMOptimizer
    prob = GSTProblem(config, seed)
    try: prob.base_model.sim = 'map'; prob.truth_model.sim = 'map'
    except Exception: pass
    sh = prob.normalize_shots(int(shots))
    dataset = prob.simulate_dataset(sh)                 # default seed=data_seed -> SAME data as ladder/fixed_fpr
    data = pygsti.protocols.ProtocolData(prob.design, dataset)
    opt = SimplerLMOptimizer(maxiter=LM_MAXITER, maxfev=LM_MAXITER, tol=1e-6,
                             init_munu='auto', oob_action='reject')
    proto = pygsti.protocols.StandardGST(modes=LM_MODES, target_model=prob.target_model,
                                         optimizer=opt, verbosity=0)
    res = proto.run(data)
    keys = list(res.estimates.keys())
    est = res.estimates[LM_MODES if LM_MODES in res.estimates else keys[0]]
    fit = est.models['final iteration estimate']
    summ, _, _ = prob.aligned_error_metrics(fit, prob.truth_model, 'truth')
    mls = list(config.max_lengths)
    stage_keys = sorted([k for k in est.models if re.fullmatch(r'iteration \d+ estimate', k)],
                        key=lambda k: int(k.split()[1]))
    traj = []
    for si, k in enumerate(stage_keys):
        try:
            st, _, _ = prob.aligned_error_metrics(est.models[k], prob.truth_model, 'truth')
            traj.append({'stage': si, 'max_length': (mls[si] if si < len(mls) else si),
                         INF: float(st[INF]), SPAM: float(st.get(SPAM, float('nan')))})
        except Exception: pass
    out = {'seed': seed, 'method': 'LM', INF: float(summ[INF]),
           SPAM: float(summ.get(SPAM, float('nan'))),
           'shots_per_circuit': int(shots), 'total_shots': int(sh.sum())}
    lm_dir.mkdir(parents=True, exist_ok=True)
    (lm_dir / 'summary.json').write_text(json.dumps(out, indent=2))
    pd.DataFrame(traj).to_csv(lm_dir / 'lm_trajectory.csv', index=False)
    return out

# ---- run/load all three per seed (ladder already produced above) ----
comp_rows = []
for seed in SEEDS:
    sd = RESULTS_DIR / f'seed_{seed:06d}'
    if (sd / 'ladder_summary.json').exists():
        lad = json.loads((sd / 'ladder_summary.json').read_text())
        comp_rows.append({'seed': seed, 'method': 'ladder', INF: float(lad[INF]),
                          'accounted_shots': float(lad.get('accounted_revealed_shots', lad.get('total_shots', float('nan')))),
                          'physical_shots': float(lad.get('total_shots', float('nan')))})
    else:
        print(f'  (seed {seed}: run the ladder cell above first)')
    ff = run_or_load_fixed_fpr(seed, SHOTS, sd / 'fixed_fpr', FORCE)
    comp_rows.append({'seed': seed, 'method': 'fixed_FPR', INF: float(ff[INF]),
                      'accounted_shots': float(ff.get('accounted_revealed_shots', float('nan'))),
                      'physical_shots': float(ff.get('physical_precomputed_shots', ff.get('accounted_revealed_shots', float('nan'))))})
    lm = fit_or_load_lm(seed, SHOTS, sd / 'lm', FORCE)
    comp_rows.append({'seed': seed, 'method': 'LM', INF: float(lm[INF]),
                      'accounted_shots': float(lm.get('total_shots', float('nan'))),
                      'physical_shots': float(lm.get('total_shots', float('nan')))})
    print(f'seed {seed} done')

comp = pd.DataFrame(comp_rows)
comp.to_csv(RESULTS_DIR / 'comparison_summary.csv', index=False)
print()
print(comp.pivot_table(index='seed', columns='method', values=INF).to_string())

### Convergence down the length ladder: POUNDERS-ladder vs LM (fixed_FPR all-at-once as reference)

In [ ]:
COL = {'ladder': '#0072B2', 'LM': '#555555', 'fixed_FPR': '#E69F00'}

def _collect(paths, seed_from):
    frames = []
    for p in paths:
        fr = pd.read_csv(p); fr['seed'] = seed_from(p); frames.append(fr)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

lad_tr = _collect(sorted(RESULTS_DIR.glob('seed_*/ladder_trajectory.csv')),
                  lambda p: int(p.parent.name.split('_')[-1]))
lm_tr  = _collect(sorted(RESULTS_DIR.glob('seed_*/lm/lm_trajectory.csv')),
                  lambda p: int(p.parents[1].name.split('_')[-1]))

fig, ax = plt.subplots(figsize=(9.5, 6))
def band(tr, label, color):
    if tr.empty or INF not in tr: return
    g = tr.groupby('max_length')[INF]
    x = np.array(sorted(tr['max_length'].unique()), dtype=float)
    med = g.median().reindex(x).to_numpy(float)
    q25 = g.quantile(0.25).reindex(x).to_numpy(float)
    q75 = g.quantile(0.75).reindex(x).to_numpy(float)
    ax.plot(x, med, 'o-', color=color, lw=2.3, ms=7, label=label)
    ax.fill_between(x, q25, q75, color=color, alpha=0.18)
band(lad_tr, 'ladder POUNDERS', COL['ladder'])
band(lm_tr, 'LM (StandardGST)', COL['LM'])
try:
    ff_med = comp[comp['method'] == 'fixed_FPR'][INF].median()
    if np.isfinite(ff_med):
        ax.axhline(ff_med, ls='--', color=COL['fixed_FPR'], lw=2.0,
                   label=f'fixed_FPR (all-at-once): {ff_med:.1e}')
except Exception: pass
ax.axhline(1e-4, ls=':', color='black', lw=1.1, label='target 1e-4')
ax.set_yscale('log')
xs = sorted(set(lad_tr.get('max_length', pd.Series(dtype=float))).union(
            set(lm_tr.get('max_length', pd.Series(dtype=float))))) or [1, 2, 4, 8, 16, 32, 64]
ax.set_xscale('log', base=2); ax.set_xticks(xs); ax.set_xticklabels([int(v) for v in xs])
ax.set_xlabel('max circuit length L (ladder stage)')
ax.set_ylabel('mean gate infidelity to truth (log)')
ax.set_title('Convergence down the length ladder (shot-matched): ladder vs LM')
ax.grid(alpha=0.25, which='both'); ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'ladder_vs_lm_vs_fixed.png', dpi=200, bbox_inches='tight')
plt.show()

### Final infidelity comparison

In [ ]:
order = ['ladder', 'fixed_FPR', 'LM']
vals = [(m, comp[comp['method'] == m][INF].dropna().to_numpy()) for m in order]
vals = [(m, v) for m, v in vals if len(v)]
fig, ax = plt.subplots(figsize=(7, 5.5))
if vals:
    try:
        bx = ax.boxplot([v for _, v in vals], tick_labels=[m for m, _ in vals], patch_artist=True)
    except TypeError:
        bx = ax.boxplot([v for _, v in vals], labels=[m for m, _ in vals], patch_artist=True)
    for b, (m, _) in zip(bx['boxes'], vals):
        b.set(facecolor=COL[m], alpha=0.45, edgecolor=COL[m])
    for md_ in bx['medians']:
        md_.set(color='black', linewidth=1.5)
ax.set_yscale('log'); ax.axhline(1e-4, ls='--', color='black', lw=1.1)
ax.set_ylabel('final infidelity to truth (log)')
ax.set_title('Final infidelity: ladder vs fixed_FPR vs LM (shot-matched)')
ax.grid(axis='y', alpha=0.25, which='both')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'final_infidelity_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print(comp.groupby('method')[INF].agg(['median', 'min', 'max', 'count']).to_string())

### Total shots used (measurement cost)

All three fit the same physical dataset (1918 x `SHOTS`), so **physical** shots are equal (the fair-comparison baseline). The **accounted** bars are the FPR-reduced cost -- the circuits you'd actually measure on hardware: **LM** has no FPR so it must measure all of them, while the **ladder** and **fixed_FPR** only measure their FPR-selected set. Lower accounted = same accuracy for fewer shots.

In [ ]:
order = ['ladder', 'fixed_FPR', 'LM']
present = [m for m in order if m in set(comp['method'])]
med_acc = comp.groupby('method')['accounted_shots'].median()
med_phy = comp.groupby('method')['physical_shots'].median()
x = np.arange(len(present)); w = 0.38
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.bar(x - w/2, [med_phy.get(m, np.nan) for m in present], w,
       label='physical (measured)', color='0.72', edgecolor='0.4')
ax.bar(x + w/2, [med_acc.get(m, np.nan) for m in present], w,
       label='accounted (FPR-reduced)',
       color=[COL.get(m, '#888888') for m in present], edgecolor='black', linewidth=0.5)
for i, m in enumerate(present):
    a = med_acc.get(m, np.nan)
    if np.isfinite(a):
        ax.text(i + w/2, a, f'{int(a):,}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(present)
ax.set_ylabel('total shots (median across seeds)')
ax.set_title('Total shots used: physical vs FPR-accounted (shot-matched)')
ax.grid(axis='y', alpha=0.25); ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'total_shots_used.png', dpi=200, bbox_inches='tight')
plt.show()
print(pd.DataFrame({'physical': med_phy, 'accounted': med_acc}).reindex(present).to_string())